In [70]:
south_korea_world_cup_xi_vs_czechia = [
    ["Kim Seung-gyu", "GK", "FC Tokyo", 72,],
    ["Kim Min-jae", "CB", "Bayern Munich", 80,],
    ["Lee Gi-hyuk", "CB", "Gangwon", 72,],
    ["Lee Han-beom", "CB", "Midtjylland", 72,],
    ["Lee Tae-seok", "LWB", "Austria Wien", 73,],
    ["Seol Young-woo", "RWB", "Red Star Belgrade", 73,],
    ["Paik Seung-ho", "CM", "Birmingham", 74,],
    ["Hwang In-beom", "CM", "Feyenoord", 77,],
    ["Lee Jae-sung", "CM", "Mainz", 76,],
    ["Lee Kang-in", "CAM", "PSG", 79,],
    ["Son Heung-min", "ST", "LAFC", 83,],

]
south_korea_world_cup_bench_vs_czechia = [
    ["Oh Hyeon-gyu", "ST", "Besiktas", 74,],
    ["Hwang Hee-chan", "CAM", "Wolves", 76,],
    ["Eom Ji-sung", "CM", "Swansea", 73,],
    ["Kim Jin-gyu ", "CM", "Jeonbuk", 74,],
    ["Park Jin-seob", "CB", "Zhejiang", 71,],
]

In [ ]:
czechia_world_cup_xi_vs_south_korea = [
    ["Matej Kovar", "GK", "PSV Eindhoven", 75,],
    ["Stepan Chaloupek", "CB", "Slavia Prague", 71,],
    ["Robin Hranac", "CB", "Hoffenheim", 73,],
    ["Ladislav Krejci", "CB", "Wolverhampton", 77,],
    ["Vladimir Coufal", "RWB", "Hoffenheim", 76,],
    ["Alexandr Sojka", "CM", "Viktoria Plzen", 73,],
    ["Tomas Soucek", "CM", "West Ham", 77,],
    ["Jaroslav Zeleny", "LWB",	"Sparta Prague", 74,],
    ["Pavel Sulc", "CAM", "Olympique Lyonnais", 79,],
    ["Lukas Provod", "CAM", "Slavia Prague", 78,],
    ["Patrik Schick" , "ST", "Bayer Leverkusen", 86,],

]
czechia_world_cup_bench_vs_south_korea = [
    ["Adam Hlozek",  "CAM", "Hoffenheim", 77,],
    ["Tomas Chory", "ST", "Slavia Prague", 76,],
    ["Mojmir Chytil", "ST", "Slavia Prague", 74,],
    ["Michal Sadilek", "CM", "Slavia Prague", 75,],

In [138]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)

 
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)

        

In [38]:
gk = GoalkeeperProfile("Kim Seung-gyu", "South Korea", "goalkeeper", 72)
gk.input_match_stats(
    minutes_played=96,
    saves=3,
    saves_inside_box=3,
    saves_outside_box=0,
    goals_conceded=1,
    xG_faced=2.07,
    goals_prevented=1.07,
    total_passes=34,
    accurate_passes=19,
    total_long_balls=23,
    accurate_long_balls=8,
    touches=39,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Kim Seung-gyu's match rating: 8.26


In [40]:
player = PlayerProfile("Lee Gi-hyuk", "South Korea", "defender", 72) 

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=58,
    total_passes=62,
    expected_goals=0,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=4,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=8,
    interceptions=3,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=5,
    duels_lost=4,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=4,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=1,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1, #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Lee Gi-hyuk's match rating: 6.98


In [42]:
player = PlayerProfile("Kim Min-Jae", "South Korea", "defender", 80)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=51,
    total_passes=54,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=3,
    interceptions=2,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=6,
    duels_lost=3,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=4,
    aerial_duels_total=5,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kim Min-Jae's match rating: 7.45


In [44]:
player = PlayerProfile("Han Boem Lee", "South Korea", "defender", 72)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=53,
    total_passes=63,
    expected_goals=0.04,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=5,
    interceptions=4,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=6,
    duels_lost=5,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=5,
    aerial_duels_total=7,
    fouled=0,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Han Boem Lee's match rating: 6.84


In [46]:
player = PlayerProfile("Tae Seok Lee", "South Korea", "defender", 73)

player.input_match_stats(
    minutes=69,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=21,
    total_passes=25,
    expected_goals=0,
    expected_assists=0.34,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=4,
    total_long_balls=5,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=6,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=4,
    duels_lost=5,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=3,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Tae Seok Lee's match rating: 7.06


In [48]:
player = PlayerProfile("Paik Seung Ho", "South Korea", "midfielder", 74)

player.input_match_stats(
    minutes=84,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=56,
    total_passes=63,
    expected_goals=0,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=4,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Paik Seung Ho's match rating: 7.36


In [50]:
player = PlayerProfile("Hwang In Beom", "South Korea", "midfielder", 77)

player.input_match_stats(
    minutes=84,
    goals=1,
    assists=1,
    total_shots=3,
    shots_on_target=2,
    accurate_passes=73,
    total_passes=81,
    expected_goals=0.91,
    expected_assists=0.5,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hwang In Beom's match rating: 10.00


In [52]:
player = PlayerProfile("Seol Young Woo", "South Korea", "defender", 73)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=30,
    total_passes=32,
    expected_goals=0,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=3,
    total_long_balls=3,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Seol Young Woo's match rating: 7.03


In [54]:
player = PlayerProfile("Lee Jae Seung", "South Korea", "midfielder", 76)

player.input_match_stats(
    minutes=62,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=25,
    total_passes=33,
    expected_goals=0.45,
    expected_assists=0.1,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=4,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=2,
    aerial_duels_total=4,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Lee Jae Seung's match rating: 5.80


In [56]:
player = PlayerProfile("Kang In Lee", "South Korea", "midfielder", 79)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=38,
    total_passes=38,
    expected_goals=0.04,
    expected_assists=0.21,
    successful_dribbles=5,
    total_dribbles=6,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=3,
    total_long_balls=3,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=2,
    duels_won=10,
    duels_lost=4,
    ground_duels_won=10,
    ground_duels_total=14,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=4,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kang In Lee's match rating: 9.82


In [58]:
player = PlayerProfile("Son Heung Min", "South Korea", "forward", 83)

player.input_match_stats(
    minutes=69,
    goals=0,
    assists=0,
    total_shots=3,
    shots_on_target=1,
    accurate_passes=20,
    total_passes=22,
    expected_goals=1.01,
    expected_assists=0.04,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=2,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Son Heung Min's match rating: 5.45


In [36]:
player = PlayerProfile("Hwang Hee Chan", "South Korea", "forward", 76)

player.input_match_stats(
    minutes=34,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=10,
    total_passes=11,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hwang Hee Chan's match rating: 7.65


In [62]:
player = PlayerProfile("Hyun Gyo Oh", "South Korea", "forward", 74)

player.input_match_stats(
    minutes=27,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=4,
    total_passes=7,
    expected_goals=0.57,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=2,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hyun Gyo Oh's match rating: 9.07


In [64]:
player = PlayerProfile("Ji Sung Eorm", "South Korea", "midfielder", 73)

player.input_match_stats(
    minutes=27,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=2,
    total_passes=3,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=3,
    duels_lost=0,
    ground_duels_won=2,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Ji Sung Eorm's match rating: 7.57


In [66]:
player = PlayerProfile("Jin Gyu Kim ", "South Jorea", "midfielder", 74)

player.input_match_stats(
    minutes=12,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=3,
    total_passes=4,
    expected_goals=0.04,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Jin Gyu Kim 's match rating: 5.89


In [68]:
player = PlayerProfile("Jin Seob Park", "South Korea", "defender", 71)

player.input_match_stats(
    minutes=12,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=6,
    total_passes=9,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=3,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Jin Seob Park's match rating: 6.90


In [ ]:
#Czechia Ratings 

In [74]:
gk = GoalkeeperProfile("Matej Kovar", "Czechia", "goalkeeper", 75)
gk.input_match_stats(
    minutes_played=96,
    saves=4,
    saves_inside_box=3,
    saves_outside_box=1,
    goals_conceded=2,
    xG_faced=2.06,
    goals_prevented=0.06,
    total_passes=26,
    accurate_passes=9,
    total_long_balls=25,
    accurate_long_balls=8,
    touches=37,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Matej Kovar's match rating: 7.54


In [96]:
player = PlayerProfile("Krejci", "Czechia", "defender", 77)

player.input_match_stats(
    minutes=96,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=31,
    total_passes=45,
    expected_goals=0.58,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=0,
    accurate_long_balls=6,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=7,
    duels_lost=6,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=3,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Krejci's match rating: 8.85


In [98]:
player = PlayerProfile("Hranac", "Czechia", "defender", 73)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=30,
    total_passes=36,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=6,
    total_long_balls=10,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=2,
    duels_lost=5,
    ground_duels_won=0,
    ground_duels_total=4,
    aerial_duels_won=2,
    aerial_duels_total=3,
    fouled=0,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hranac's match rating: 6.66


In [100]:
player = PlayerProfile("Chaloupek", "Czechia", "defender", 71)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=18,
    total_passes=29,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=9,
    dispossessed=2,
    tackles_won=1,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=2,
    duels_lost=6,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Chaloupek's match rating: 6.23


In [102]:
player = PlayerProfile("Zeleny", "Czechia", "midfielder", 74)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=22,
    total_passes=28,
    expected_goals=0,
    expected_assists=0.04,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=5,
    interceptions=0,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Zeleny's match rating: 8.49


In [104]:
player = PlayerProfile("Sojka", "Czechia", "midfielder", 73)

player.input_match_stats(
    minutes=84,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=26,
    total_passes=33,
    expected_goals=0,
    expected_assists=0.17,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=3,
    total_long_balls=4,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=2,
    ball_recoveries=3,
    dribbled_past=3,
    duels_won=3,
    duels_lost=4,
    ground_duels_won=3,
    ground_duels_total=7,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sojka's match rating: 7.89


In [106]:
player = PlayerProfile("Soucek", "Czechia", "midfielder", 77)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=25,
    total_passes=34,
    expected_goals=0.28,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=3,
    total_long_balls=5,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=2,
    duels_lost=5,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=2,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Soucek's match rating: 6.00


In [108]:
player = PlayerProfile("Coufal", "Czechia", "defender", 76)

player.input_match_stats(
    minutes=96,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=26,
    expected_goals=0,
    expected_assists=0.08,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=6,
    accurate_long_balls=0,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=2,
    duels_lost=6,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=1,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Coufal's match rating: 7.30


In [120]:
player = PlayerProfile("Pavel Sulc", "Czechia", "forward", 79)

player.input_match_stats(
    minutes=64,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=13,
    total_passes=18,
    expected_goals=0.01,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=7,
    dribbled_past=1,
    duels_won=5,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0, #Just for Defenders
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Pavel Sulc's match rating: 7.02


In [122]:
player = PlayerProfile("Provod", "Czechia", "midfielder", 77)

player.input_match_stats(
    minutes=64,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=22,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=4,
    duels_lost=4,
    ground_duels_won=4,
    ground_duels_total=5,
    aerial_duels_won=0,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0, #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Provod's match rating: 7.25


In [124]:
player = PlayerProfile("Schick", "Czechia", "forward", 86)

player.input_match_stats(
    minutes=64,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=3,
    total_passes=5,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=4,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Schick's match rating: 5.92


In [128]:
player = PlayerProfile("Hlozek", "Czechia", "midfielder", 77)

player.input_match_stats(
    minutes=32,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=4,
    total_passes=4,
    expected_goals=0.75,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hlozek's match rating: 6.20


In [132]:
player = PlayerProfile("Chory", "Czechia", "forward", 76)

player.input_match_stats(
    minutes=32,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=8,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=5,
    duels_lost=6,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=4,
    aerial_duels_total=9,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Chory's match rating: 6.05


In [134]:
player = PlayerProfile("Sadilek", "Czechia", "midfielder", 75)

player.input_match_stats(
    minutes=32,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=2,
    accurate_passes=8,
    total_passes=11,
    expected_goals=0.15,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sadilek's match rating: 6.00


In [136]:
player = PlayerProfile("Chytil", "Czechia", "forward", 74)

player.input_match_stats(
    minutes=12,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=2,
    total_passes=2,
    expected_goals=0,
    expected_assists=0.06,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Chytil's match rating: 5.70
